# Chapter 12 — Serial Correlation and Heteroscedasticity in Time Series
## University of Neuchâtel — Python for Econometrics

Topics:
1. What is serial correlation and its consequences
2. Detecting serial correlation: Durbin-Watson and Breusch-Godfrey tests
3. Correcting with HAC (Newey-West) standard errors
4. Cochrane-Orcutt GLS procedure

**Format:** One cell per step. Manual calculation first, then statsmodels, then `np.isclose()`.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
import statsmodels.stats.diagnostic as diag
from statsmodels.graphics.tsaplots import plot_acf
from scipy import stats
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — What is Serial Correlation?

**Serial correlation** (autocorrelation) occurs when regression errors are correlated over time:

$$Corr(u_t, u_s) \neq 0 \quad \text{for } t \neq s$$

**AR(1) errors model:** $u_t = \rho u_{t-1} + e_t$ where $|\rho| < 1$

**Consequences of serial correlation:**
- OLS estimator $\hat{\beta}$ is still **unbiased** and **consistent**
- But OLS standard errors are **biased** → $t$- and $F$-tests are **invalid**
- With positive autocorrelation ($\rho > 0$): OLS SEs are **too small** (over-reject $H_0$)

**When does it arise?** Almost always in time series regressions.

## 2 — Load Data: `phillips` Dataset

In [ ]:
df = woo.dataWoo('phillips')
print('Shape:', df.shape)
print(df[['year','inf','unem']].head(10))

In [ ]:
print('Summary statistics:')
print(df[['inf','unem']].describe().round(3))

## 3 — Baseline OLS: inf ~ unem

### Step 3.1 — Manual OLS

In [ ]:
n = len(df)
x = df['unem'].values
y = df['inf'].values
b1_man = np.sum((x - x.mean())*(y - y.mean())) / np.sum((x - x.mean())**2)
b0_man = y.mean() - b1_man * x.mean()
print(f'Manual OLS: inf ~ unem')
print(f'  b0 = {b0_man:.4f}')
print(f'  b1 = {b1_man:.4f}')

### Step 3.2 — Statsmodels OLS

In [ ]:
res = smf.ols('inf ~ unem', data=df).fit()
print(res.summary())

### Step 3.3 — Compare manual vs statsmodels

In [ ]:
print(f'b0: manual={b0_man:.4f}, sm={res.params["Intercept"]:.4f}, match={np.isclose(b0_man, res.params["Intercept"])}')
print(f'b1: manual={b1_man:.4f}, sm={res.params["unem"]:.4f}, match={np.isclose(b1_man, res.params["unem"])}')

### Step 3.4 — Plot residuals over time

In [ ]:
e = res.resid.values
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(df['year'], e, linewidth=1.5, marker='o', markersize=3)
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Residual')
ax.set_title('OLS Residuals: inf ~ unem (do they show a pattern?)')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch12_01_residuals.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
plot_acf(e, lags=10, ax=ax)
ax.set_title('ACF of OLS Residuals')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch12_02_acf_resid.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4 — Durbin-Watson Test

**Manual formula:**

$$DW = \frac{\sum_{t=2}^T (\hat{u}_t - \hat{u}_{t-1})^2}{\sum_{t=1}^T \hat{u}_t^2}$$

- $DW \approx 2$: no serial correlation
- $DW < 2$: positive autocorrelation (common in economic time series)
- $DW > 2$: negative autocorrelation

### Step 4.1 — DW statistic manually

In [ ]:
e_diff = np.diff(e)  # e_t - e_{t-1}
DW_manual = np.sum(e_diff**2) / np.sum(e**2)
print(f'Durbin-Watson statistic (manual): {DW_manual:.4f}')
if DW_manual < 1.5:
    print('  DW < 1.5: evidence of POSITIVE serial correlation')
elif DW_manual > 2.5:
    print('  DW > 2.5: evidence of NEGATIVE serial correlation')
else:
    print('  DW ≈ 2: no strong evidence of serial correlation')

### Step 4.2 — DW with statsmodels

In [ ]:
DW_sm = durbin_watson(e)
print(f'Durbin-Watson (statsmodels): {DW_sm:.4f}')
print(f'Match: {np.isclose(DW_manual, DW_sm)}')

## 5 — Breusch-Godfrey Test

**More powerful than DW.** Tests for AR(p) serial correlation in residuals.

**$H_0$:** No serial correlation up to order $p$

**Manual procedure:**
1. Get OLS residuals $\hat{u}_t$
2. Regress $\hat{u}_t$ on original regressors and lagged residuals $\hat{u}_{t-1}, ..., \hat{u}_{t-p}$
3. $LM = n \cdot R^2$ from auxiliary regression
4. $p$-value from $\chi^2(p)$ distribution

### Step 5.1 — BG test manually (p=2)

In [ ]:
p_bg = 2
df_bg = df[['inf','unem']].copy()
df_bg['e'] = e
df_bg['e_lag1'] = df_bg['e'].shift(1)
df_bg['e_lag2'] = df_bg['e'].shift(2)
df_bg = df_bg.dropna()
print('BG auxiliary regression data (first 5 rows):')
print(df_bg.head())

In [ ]:
res_bg_aux = smf.ols('e ~ unem + e_lag1 + e_lag2', data=df_bg).fit()
n_bg = len(df_bg)
LM_manual = n_bg * res_bg_aux.rsquared
from scipy.stats import chi2
pval_bg_manual = chi2.sf(LM_manual, df=p_bg)
print(f'BG LM statistic (manual): {LM_manual:.4f}')
print(f'p-value (manual):         {pval_bg_manual:.4f}')
if pval_bg_manual < 0.05:
    print('  Reject H0: serial correlation detected at 5% level')
else:
    print('  Fail to reject H0: no serial correlation')

### Step 5.2 — BG test with statsmodels

In [ ]:
X_exog = sm.add_constant(df['unem'].values)
bg_test = diag.acorr_breusch_godfrey(res, nlags=p_bg)
print('BG test (statsmodels):')
print(f'  LM statistic: {bg_test[0]:.4f}')
print(f'  p-value:      {bg_test[1]:.4f}')

In [ ]:
print(f'LM match: manual={LM_manual:.4f}, sm={bg_test[0]:.4f}')
print(f'Close? {np.isclose(LM_manual, bg_test[0], rtol=0.01)}')

## 6 — HAC (Newey-West) Standard Errors

**Fix for serial correlation:** Use **Heteroscedasticity and Autocorrelation Consistent (HAC)**
standard errors (also called Newey-West SEs).

OLS coefficients remain the same — only the standard errors change.

HAC SEs are larger when there is positive autocorrelation.

### Step 6.1 — Robust (HAC) SEs with statsmodels

In [ ]:
res_hac = res.get_robustcov_results(cov_type='HAC', maxlags=2)
print('HAC standard errors (Newey-West, maxlags=2):')
print(res_hac.summary())

### Step 6.2 — Compare OLS vs HAC standard errors

In [ ]:
se_ols = res.bse
se_hac = res_hac.bse
print('Comparison: OLS vs HAC standard errors')
print(f'  Intercept: OLS SE={se_ols["Intercept"]:.4f}, HAC SE={se_hac["Intercept"]:.4f}')
print(f'  unem:      OLS SE={se_ols["unem"]:.4f}, HAC SE={se_hac["unem"]:.4f}')
ratio = se_hac['unem'] / se_ols['unem']
print(f'  HAC/OLS ratio for unem: {ratio:.3f}')
if ratio > 1:
    print('  HAC SEs are larger → OLS t-stats were over-stated')

## 7 — Cochrane-Orcutt GLS Procedure

**Idea:** If $u_t = \rho u_{t-1} + e_t$, we can transform the model to remove autocorrelation.

**Steps:**
1. Estimate $\rho$ from lagged residuals
2. Quasi-difference all variables: $\tilde{y}_t = y_t - \hat{\rho} y_{t-1}$
3. Run OLS on quasi-differenced data
4. Repeat until convergence

### Step 7.1 — Estimate rho manually

In [ ]:
rho_hat = np.corrcoef(e[1:], e[:-1])[0,1]
print(f'Estimated rho (AR(1) coefficient): {rho_hat:.4f}')
print('(Positive rho confirms positive serial correlation)')

### Step 7.2 — Quasi-difference y and X

In [ ]:
y_orig = df['inf'].values
x_orig = df['unem'].values
# Quasi-differenced values: y_t - rho * y_{t-1}
y_qd = y_orig[1:] - rho_hat * y_orig[:-1]
x_qd = x_orig[1:] - rho_hat * x_orig[:-1]
print(f'Original data length: {len(y_orig)}')
print(f'Quasi-differenced length: {len(y_qd)} (lose one obs)')
print('First 5 quasi-differenced y values:', y_qd[:5].round(4))

### Step 7.3 — OLS on quasi-differenced data (manual)

In [ ]:
b1_co = np.sum((x_qd - x_qd.mean())*(y_qd - y_qd.mean())) / np.sum((x_qd - x_qd.mean())**2)
b0_co = y_qd.mean() - b1_co * x_qd.mean()
print(f'Cochrane-Orcutt estimates (manual):')
print(f'  b0 = {b0_co:.4f}')
print(f'  b1 = {b1_co:.4f}')

### Step 7.4 — Cochrane-Orcutt via statsmodels GLSAR

In [ ]:
from statsmodels.regression.linear_model import GLSAR
X_sm = sm.add_constant(df['unem'].values)
res_co_sm = GLSAR(df['inf'].values, X_sm, rho=1).iterative_fit(maxiter=10)
print('Cochrane-Orcutt (GLSAR) results:')
print(f'  Intercept: {res_co_sm.params[0]:.4f}')
print(f'  unem:      {res_co_sm.params[1]:.4f}')
print(f'  Estimated rho: {res_co_sm.model.rho:.4f}')

### Step 7.5 — Summary: OLS vs HAC vs CO

In [ ]:
print('Comparison: unem coefficient')
print(f'  OLS:             b1={res.params["unem"]:.4f},  SE={res.bse["unem"]:.4f}')
print(f'  OLS+HAC:         b1={res_hac.params["unem"]:.4f},  SE={res_hac.bse["unem"]:.4f}')
print(f'  Cochrane-Orcutt: b1={res_co_sm.params[1]:.4f},  SE={res_co_sm.bse[1]:.4f}')

In [ ]:
sumtab = pd.DataFrame({
    'Method': ['OLS', 'OLS+HAC', 'Cochrane-Orcutt'],
    'b0': [res.params['Intercept'], res_hac.params['Intercept'], res_co_sm.params[0]],
    'b1_unem': [res.params['unem'], res_hac.params['unem'], res_co_sm.params[1]],
    'SE_b1': [res.bse['unem'], res_hac.bse['unem'], res_co_sm.bse[1]]
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch12_summary.csv'), index=False)
print('Saved table_ch12_summary.csv')